In [13]:
from typing import Annotated
from dotenv import load_dotenv
load_dotenv()

from typing_extensions import TypedDict
from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

In [29]:
llm = init_chat_model("google_genai:gemini-3.5-flash")

class State(TypedDict):
     messages: Annotated[list, add_messages]

def chatbot(state: State) -> State:
    return {
        "messages": [llm.invoke(state["messages"])]
    }

builder = StateGraph(State)
builder.add_node("chatbot_node", chatbot)

builder.add_edge(START, "chatbot_node")
builder.add_edge("chatbot_node", END)

graph = builder.compile()

In [30]:
message = {"role": "user", "content": "Who walked on the moon for the first time? Print only the name"}
# message = {"role": "user", "content": "What is the latest price of MSFT stock?"}
response = graph.invoke({"messages":[message]})

response["messages"]

[HumanMessage(content='Who walked on the moon for the first time? Print only the name', additional_kwargs={}, response_metadata={}, id='2a81e36f-deea-42c8-99c7-236ff70a9fa2'),
 AIMessage(content=[{'type': 'text', 'text': 'Neil Armstrong', 'extras': {'signature': 'Et4CCtsCARFNMg+FtfNs0RlP5rmCRpRcOKlRZ/noPQ0N8ujBair8MXmDzyAQBGxOH3kEyZdPFw/ANdBJeyVHVlVGrNl72F6o6fxKLs5MLlcmYt7Ux5ek3WsOwNhZwyqwGSCoVZLPl96tfpZ8LIadQLQDxlVbZI/OWqbgbXnt7UnbI1YMdFmTocvde5webQq4OggSRwefMM1Ut2+JEFisrtVvYzkM6Dc8ULB/Ynzua0hzYcZJS3CUi9y/TI5WJty6Bdxd1+hlkq/vt7HSMwjPvzEcYwuUlpBSqVmyH0zMVejOvd9oCxX2IrwQ2mS1/3f8AZ2FJtQ16ua8vCmxcFSFWM/ISh+m0L+SB4E//YQ9yVV5NlwVwHfI2YD49CyATsPwXOoFjCR3IRH3b9hzWTk/Gnolg7LZ6Z9zZlxkkW2r6/xUmXre7xzYl5D6tjJP6xM5lXZCXFlAAK1kcNjzgEneq78='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f601e-bad9-76a1-8b69-c8969a5b6233-0', tool_calls=[], invalid_tool_calls=[], usage_meta

In [31]:
state = None
while True:
    in_message = input("You: ")
    if in_message.lower() in {"quit","exit"}:
        break
    if state is None:
        state: State = {
            "messages": [{"role": "user", "content": in_message}]
        }
    else:
        state["messages"].append({"role": "user", "content": in_message})

    state = graph.invoke(state)
    print("Bot:", state["messages"][-1].content)

Bot: [{'type': 'text', 'text': 'Hello! How can I help you today?', 'extras': {'signature': 'EtwFCtkFARFNMg+Ib7QXfp63pZe6VK+CAuxOzaRP47DhR8069HeqNimZEBbXxOLAa+OY9Yx8OUaWCNkuhZnQtXZV5HUHE1Qd3xfe+5kA+BtKsQuf9qDjOBYP6omn06HxfOfcyjOedHXk74sWtOOEhwf8nEky78VOh+Sz416UkL3L4PL+UEOL4wVMKT+T2I6gfY1XfBu2N/Sg6NCIa6oCMrxJ5goZKXct6XuXL2VU0ybcEYk1bHcGK9KGjwz0Lfed+ewNFbLTc1M8cHsuFm/b5kSoD0u2n5QhQQITxA9J1/vqIExyr7kqA7U9/I5bc6JagplaKrmALbbb8BXp+mmE28likjgqad22PHgXAifyr0KDWcPT6uDRwOvQJkelpbyt0cKT9tk7ABWy8K8ywY14L2jNyye1RMNxfKvCMSldCN1rWZLYK8MwZGt2Qek55EeT8vra9/jDUcGifvnbQatGvFfPXGBKV/yEsd1f8ZGYoDXF+rTpYbpeS40hXdlGv4OCBB2R1ZFNoL8bySl2azBWwzQSsIMDjgUD3pGR9XNU7wPZPqxPBvO50p7i/1KdtxjWOVqV4UZ1ne/eecjE9YXsNnoIn+7hAdXZxNij+v2iDBYNWwcENMmlIL10Py7Pl3U4rxUycV5W4WJaDZe4dMQQIZg9xAO0IS9Uokitbb3bn8xvbSSpO0GBZV8WQoq6LH6Em09pwb6/yBRPF3AH2adIvWxzLV/dH1cVbFGMMLBNuCLiUflLX/em7Epuy/B1XYZTm20JvXQWP4DnPOp6icRlV2JNybBU+OwO2LTf6SmAxCiUUHXF8AqM0mTCmsGeIT6g7IY/nu9sH4xaD4xoQLoRwWle5O7VVnDJSmWB/onb2shT7TIhc4zho2CFaxCGh9DTmj3/U3YCFATv